# Smart Sentence Finder

This notebook demonstrates the full sentence-search workflow on the current default embedding models.


## Objectives

The goals of this notebook are to:

- load and clean a source text document
- rank the most relevant sentences for a query across several embedding models
- benchmark those models with the current silhouette-based workflow
- generate a presentation chart from the benchmark results
- summarize the main findings at the end


## Imports

Import the shared project utilities, configure notebook progress behavior, and define the core paths and runtime settings.


In [ ]:
from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path

import torch
from IPython.display import Image, SVG, display

ROOT = Path.cwd().resolve()
SRC = ROOT / "src"
if not (SRC / "smart_sentence_finder").exists():
    raise FileNotFoundError("Run this notebook from the repository root so /src is available.")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from smart_sentence_finder.cli import DEFAULT_MODELS
from smart_sentence_finder.notebook_utils import (
    build_benchmark_rows,
    configure_notebook_progress,
    get_runtime_info,
    prepare_text_data,
    rank_models_for_query,
    save_benchmark_outputs,
    save_rank_results,
)

progress_info = configure_notebook_progress()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

DATA_FILE = ROOT / "data" / "alice_in_wonderland.txt"
QUERY = "She wonders about things."
MODELS = list(DEFAULT_MODELS)

CHARS_PER_CHUNK = 10_000
TOP_N = 5
RANK_BATCH_SIZE = 32
BENCHMARK_BATCH_SIZE = 16
MAX_LENGTH = 512
MAX_SENTENCES = 1_000
K_MIN = 2
K_MAX = 10

OUTPUT_DIR = ROOT / "output" / "notebook"
PRESENTATION_DIR = ROOT / "output" / "presentation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PRESENTATION_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data file: {DATA_FILE}")
print(f"Query: {QUERY}")
print(f"Models: {len(MODELS)}")
print(f"Progress backend: {progress_info['tqdm_backend']}")


## Runtime Check

Review the notebook environment, device selection, and current model configuration before running the heavier cells.


In [ ]:
runtime_info = get_runtime_info(DEVICE)

print(f"Repo root: {ROOT}")
print(f"Python: {runtime_info['python']}")
print(f"Torch: {runtime_info['torch']}")
print(f"Device: {runtime_info['device']}")
print(f"GPU: {runtime_info['gpu']}")
print(f"FlashAttention: {runtime_info['flash_attn']}")
print(f"ipywidgets installed: {runtime_info['ipywidgets_installed']}")
print(f"Notebook progress backend: {progress_info['tqdm_backend']}")
print(f"Suppressed hub progress: {progress_info['library_progress_suppressed']['huggingface_hub']}")
print(f"Suppressed transformers progress: {progress_info['library_progress_suppressed']['transformers']}")
print(f"HF token present: {runtime_info['hf_token_present']}")
print(f"SSF_EMBED_BACKEND: {runtime_info['embed_backend']}")
print("Current default models:")
for model_name in runtime_info['models']:
    print(f"- {model_name}")


## Load And Clean Data

Load the source text, normalize it, segment it into sentences, and build the cleaned sentence set used by the ranking and benchmark workflows.


In [ ]:
prepared = prepare_text_data(DATA_FILE, chars_per_chunk=CHARS_PER_CHUNK)

print(f"Total characters: {len(prepared.text):,}")
print(f"Total cleaned sentences: {len(prepared.sentences):,}")
print(f"Rankable sentences (>5 words): {len(prepared.rankable_sentences):,}")
print("\nSample cleaned sentences:")
for sentence in prepared.rankable_sentences[:5]:
    print(f"- {sentence}")


## Process Sentences

Encode the query and sentence set with each model, rank the results by cosine similarity, and save the ranking outputs.


In [ ]:
top1_rows, rank_rows, rank_payload = rank_models_for_query(
    MODELS,
    QUERY,
    prepared.sentences,
    top_n=TOP_N,
    batch_size=RANK_BATCH_SIZE,
    max_length=MAX_LENGTH,
)

for row in top1_rows:
    print(row["model_name"])
    print(f"  score={row['score']:.4f}")
    print(f"  sentence={row['sentence']}\n")

rank_json, rank_csv = save_rank_results(
    OUTPUT_DIR,
    DATA_FILE,
    RUN_TS,
    QUERY,
    MODELS,
    rank_payload,
    rank_rows,
)

print(f"Saved ranking JSON to {rank_json}")
print(f"Saved ranking CSV to {rank_csv}")


## Benchmark Models

Run the current silhouette benchmark across the default model list, save the benchmark outputs, and generate the presentation chart assets.


In [ ]:
benchmark_rows = build_benchmark_rows(
    MODELS,
    prepared.rankable_sentences,
    max_sentences=MAX_SENTENCES,
    batch_size=BENCHMARK_BATCH_SIZE,
    max_length=MAX_LENGTH,
    k_min=K_MIN,
    k_max=K_MAX,
)

benchmark_json, benchmark_csv, chart_paths = save_benchmark_outputs(
    OUTPUT_DIR,
    PRESENTATION_DIR,
    DATA_FILE,
    RUN_TS,
    benchmark_rows,
)

for row in benchmark_rows:
    print(
        f"{row['model_name']} | silhouette={row['silhouette']:.4f} | "
        f"per_Mparam={row['silhouette_per_million_params']:.6e} | "
        f"best_k={row['best_k']} | n={row['n_sentences']} | dim={row['dim']} | "
        f"params={row['param_count'] / 1_000_000:.2f}M"
    )

print(f"\nSaved benchmark JSON to {benchmark_json}")
print(f"Saved benchmark CSV to {benchmark_csv}")
print(f"Saved benchmark SVG to {chart_paths.svg}")
print(f"Saved benchmark PNG to {chart_paths.png or 'not generated in this environment'}")


## Results

Display the benchmark comparison chart and review the main exported artifacts produced by the notebook.


In [ ]:
if chart_paths.png is not None and chart_paths.png.exists():
    display(Image(filename=str(chart_paths.png)))
else:
    display(SVG(filename=str(chart_paths.svg)))

print(f"Ranking JSON: {rank_json}")
print(f"Ranking CSV: {rank_csv}")
print(f"Benchmark JSON: {benchmark_json}")
print(f"Benchmark CSV: {benchmark_csv}")
print(f"Chart SVG: {chart_paths.svg}")
print(f"Chart PNG: {chart_paths.png or 'not generated in this environment'}")


## Key Findings

Summarize the strongest benchmark result and the top ranked sentence returned by each model.


In [ ]:
best_benchmark = benchmark_rows[0]
print(
    f"Best silhouette model: {best_benchmark['model_name']} "
    f"with silhouette={best_benchmark['silhouette']:.4f} and best_k={best_benchmark['best_k']}"
)

print("\nTop ranked sentence by model:")
for row in top1_rows:
    print(f"- {row['model_name']}: {row['score']:.4f} | {row['sentence']}")

print("\nTop 3 benchmark models:")
for row in benchmark_rows[:3]:
    print(f"- {row['model_name']}: silhouette={row['silhouette']:.4f}")
